# Kapitel 11 – pandas zur Analyse historischer Daten

In diesem Notebook lernen wir die Python-Bibliothek **pandas** kennen – das Standardwerkzeug für tabellenbasierte Datenanalyse. Wir arbeiten mit einem synthetischen Datensatz römischer **Meilensteine** (*miliaria*), der typische Fragestellungen der historischen Datenanalyse abbildet.

**Lernziele:**
1. CSV-Daten einlesen und erste Übersichten erstellen
2. Daten filtern, sortieren und auswählen
3. Daten gruppieren und aggregieren
4. Tabellen zusammenführen (`merge`)
5. Pivot-Tabellen erstellen
6. Daten visualisieren

---

## 1. Einführung: Daten einlesen und inspizieren

Wir importieren `pandas` (konventionell als `pd` abgekürzt) und laden unseren Datensatz. Dieser enthält 200 (fiktive, aber historisch plausible) Einträge römischer Meilensteine mit Informationen zu Provinz, Stadt, Kaiser, Datierung, Material, Zustand und Strassentyp.

In [ ]:
import pandas as pd

# Datensatz einlesen
df = pd.read_csv('data/miliaria_romana.csv')

# Dimensionen: (Zeilen, Spalten)
print(f"Dimensionen: {df.shape}")
print(f"Spalten: {list(df.columns)}")

In [ ]:
# Die ersten 5 Zeilen anzeigen
df.head()

In [ ]:
# Übersicht über Spalten, Datentypen und fehlende Werte
df.info()

In [ ]:
# Statistische Kennzahlen für numerische Spalten
df.describe()

### Series und DataFrame

Ein **DataFrame** ist eine zweidimensionale Tabelle. Jede einzelne Spalte ist eine **Series** – eine eindimensionale Datenstruktur mit Index.

In [ ]:
# Eine einzelne Spalte ist eine Series
kaiser_series = df['kaiser']
print(type(kaiser_series))
print(f"\nAnzahl einzigartiger Kaiser: {kaiser_series.nunique()}")
print(f"\nAlle Kaiser im Datensatz:")
print(kaiser_series.unique())

In [ ]:
# Häufigkeiten zählen
df['kaiser'].value_counts()

**Übung 1:** Verwende `.value_counts()`, um herauszufinden, welche **Provinz** die meisten Meilensteine hat.

In [ ]:
# Deine Lösung hier:


---
## 2. Datenzugriff, Auswahl und Filterung

Wir können Zeilen und Spalten auf verschiedene Weisen auswählen:
- **Spaltenauswahl**: `df['spalte']` oder `df[['spalte1', 'spalte2']]`
- **Boolean Indexing**: `df[bedingung]` – filtert Zeilen nach einer Bedingung
- **`.loc[]`**: Zugriff über Labels
- **`.iloc[]`**: Zugriff über Position (Ganzzahl)

In [ ]:
# Alle Meilensteine von Traianus
traian = df[df['kaiser'] == 'Traianus']
print(f"Anzahl Meilensteine von Traianus: {len(traian)}")
traian[['id', 'stadt', 'provinz', 'jahr', 'material']].head()

In [ ]:
# Kombinierte Bedingungen: Marmor UND gut erhalten
marmor_gut = df[(df['material'] == 'Marmor') & (df['zustand'] == 'gut erhalten')]
print(f"Marmor + gut erhalten: {len(marmor_gut)} Stück")
marmor_gut.head()

In [ ]:
# Meilensteine vor Christi Geburt
vor_chr = df[df['jahr'] < 0]
print(f"Meilensteine v. Chr.: {len(vor_chr)}")
vor_chr[['id', 'stadt', 'kaiser', 'jahr']].sort_values('jahr')

In [ ]:
# Sortieren: älteste Meilensteine zuerst
df.sort_values('jahr').head(10)[['id', 'provinz', 'stadt', 'kaiser', 'jahr']]

**Übung 2:** Finde alle Meilensteine aus **Britannia**, die aus **Sandstein** bestehen. Wie viele sind es?

In [ ]:
# Deine Lösung hier:


---
## 3. Gruppieren und Aggregieren

Die Methode `.groupby()` erlaubt es, Daten nach einer Kategorie zu gruppieren und dann statistische Kennzahlen zu berechnen. Das Prinzip: **Split – Apply – Combine**.

In [ ]:
# Anzahl Meilensteine pro Provinz
df.groupby('provinz')['id'].count().sort_values(ascending=False)

In [ ]:
# Durchschnittliche Höhe pro Material
df.groupby('material')['hoehe_cm'].mean().round(1)

In [ ]:
# Mehrere Aggregationen gleichzeitig
kaiser_stats = df.groupby('kaiser').agg(
    anzahl=('id', 'count'),
    mittlere_hoehe=('hoehe_cm', 'mean'),
    mittlere_distanz=('distanz_meilen', 'mean'),
    aeltester=('jahr', 'min'),
    juengster=('jahr', 'max')
).round(1).sort_values('anzahl', ascending=False)

kaiser_stats

> **Zur Interpretation:** Wenn eine Provinz mehr Meilensteine aufweist als eine andere, kann das verschiedene Ursachen haben: tatsächlich höhere Bautätigkeit, bessere Erhaltungsbedingungen, intensivere archäologische Erforschung – oder einfach Zufall der Überlieferung. Quantitative Befunde sind immer quellenkritisch zu interpretieren.

**Übung 3:** Berechne die durchschnittliche Anzahl **Textzeilen** pro **Zustand** (`zustand`). Welcher Zustand hat die meisten Textzeilen?

In [ ]:
# Deine Lösung hier:


---
## 4. Daten zusammenführen: `merge`

In der Praxis liegen Informationen oft in verschiedenen Tabellen vor. Die Funktion `pd.merge()` verknüpft zwei DataFrames über eine gemeinsame Spalte – ähnlich wie ein JOIN in SQL.

In [ ]:
# Zweite Tabelle: Strassentypen mit Beschreibung und Breite
strassen = pd.read_csv('data/strassentypen.csv')
strassen

In [ ]:
# Merge: Strasseninformationen zu Meilensteinen hinzufügen
df_merged = pd.merge(df, strassen, on='strassentyp', how='left')
df_merged[['id', 'stadt', 'strassentyp', 'beschreibung', 'breite_m']].head(10)

In [ ]:
# Durchschnittliche Distanz pro Strassenbreite
df_merged.groupby('strassentyp')[['distanz_meilen', 'breite_m']].mean().round(1)

---
## 5. Pivot-Tabellen

Pivot-Tabellen kreuztabellieren zwei Kategorien und berechnen für jede Kombination einen aggregierten Wert.

In [ ]:
# Kreuztabelle: Meilensteine pro Provinz und Material
pd.crosstab(df['provinz'], df['material'])

In [ ]:
# Pivot-Tabelle: Durchschnittliche Höhe pro Kaiser und Material
pivot = pd.pivot_table(
    df, 
    values='hoehe_cm', 
    index='kaiser', 
    columns='material', 
    aggfunc='mean'
).round(1)

pivot

**Übung 4:** Erstelle eine Kreuztabelle (`pd.crosstab`), die zeigt, wie viele Meilensteine pro **Kaiser** und **Zustand** existieren.

In [ ]:
# Deine Lösung hier:


---
## 6. Visualisierung

pandas bietet über `.plot()` eine direkte Schnittstelle zu `matplotlib`. Damit lassen sich Daten schnell und anschaulich darstellen.

In [ ]:
import matplotlib.pyplot as plt

# Balkendiagramm: Meilensteine pro Kaiser
fig, ax = plt.subplots(figsize=(10, 5))
df['kaiser'].value_counts().sort_values().plot(
    kind='barh', ax=ax, color='#8B4513'
)
ax.set_title('Anzahl Meilensteine pro Kaiser')
ax.set_xlabel('Anzahl')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# Materialverteilung als Kreisdiagramm
fig, ax = plt.subplots(figsize=(7, 7))
df['material'].value_counts().plot(
    kind='pie', ax=ax, autopct='%1.1f%%',
    colors=['#D2B48C', '#C0C0C0', '#F5F5DC', '#808080', '#2F4F4F']
)
ax.set_title('Verteilung der Materialien')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# Zeitliche Verteilung der Meilensteine
fig, ax = plt.subplots(figsize=(12, 4))

# Wir gruppieren die Jahre in 25-Jahres-Intervalle
df['zeitraum'] = (df['jahr'] // 25) * 25
zeitverteilung = df.groupby('zeitraum')['id'].count()

zeitverteilung.plot(kind='bar', ax=ax, color='#4682B4', edgecolor='black')
ax.set_title('Zeitliche Verteilung der Meilensteine (25-Jahres-Intervalle)')
ax.set_xlabel('Zeitraum (Beginn des Intervalls)')
ax.set_ylabel('Anzahl')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Hilfsspalte wieder entfernen
df.drop(columns='zeitraum', inplace=True)

In [ ]:
# Gestapeltes Balkendiagramm: Zustand pro Provinz
fig, ax = plt.subplots(figsize=(12, 6))
pd.crosstab(df['provinz'], df['zustand']).plot(
    kind='bar', stacked=True, ax=ax,
    colormap='YlOrBr'
)
ax.set_title('Erhaltungszustand der Meilensteine nach Provinz')
ax.set_xlabel('')
ax.set_ylabel('Anzahl')
ax.legend(title='Zustand', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

**Übung 5:** Erstelle ein Balkendiagramm, das die **Anzahl Meilensteine pro Provinz** zeigt. Wähle `kind='barh'` für ein horizontales Diagramm.

In [ ]:
# Deine Lösung hier:


---
## Zusammenfassung

In diesem Notebook haben wir die Grundoperationen von `pandas` kennengelernt:

| Operation | Funktion / Methode |
|-----------|--------------------|
| Einlesen | `pd.read_csv()` |
| Übersicht | `.head()`, `.info()`, `.describe()`, `.shape` |
| Auswählen | `df['spalte']`, `df[['sp1', 'sp2']]` |
| Filtern | `df[bedingung]`, `.loc[]`, `.iloc[]` |
| Sortieren | `.sort_values()` |
| Häufigkeiten | `.value_counts()`, `.nunique()` |
| Gruppieren | `.groupby()`, `.agg()` |
| Zusammenführen | `pd.merge()`, `pd.concat()` |
| Pivot-Tabellen | `pd.crosstab()`, `pd.pivot_table()` |
| Visualisierung | `.plot()` mit `matplotlib` |

**Merke:** Die Daten erzählen nie die ganze Geschichte. Sie sind immer das Produkt von Überlieferungsprozessen, Sammlungsentscheidungen und methodischen Vorannahmen. Die Werkzeuge helfen, Muster zu erkennen – die *Interpretation* bleibt die Aufgabe der Historiker:in.